In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base_path = "/content/drive/MyDrive/files"

folders = [f for f in os.listdir(base_path) if f.startswith("S")]

filenumber = ['03','07','11']
data = []

for folder in sorted(folders):
    folder_path = os.path.join(base_path, folder)

    if not os.path.isdir(folder_path):
        continue

    files = os.listdir(folder_path)

    for f in files:
        if f.lower().endswith(".edf"):

            run_part = f.split("R")[-1].split(".")[0]

            if run_part in filenumber:
                full_path = os.path.join(folder_path, f)
                data.append(full_path)

# Print final single list
print(data)
print("\nTotal files:", len(data))


['/content/drive/MyDrive/files/S071/S071R03.edf', '/content/drive/MyDrive/files/S071/S071R11.edf', '/content/drive/MyDrive/files/S071/S071R07.edf', '/content/drive/MyDrive/files/S072/S072R11.edf', '/content/drive/MyDrive/files/S072/S072R07.edf', '/content/drive/MyDrive/files/S072/S072R03.edf', '/content/drive/MyDrive/files/S074/S074R07.edf', '/content/drive/MyDrive/files/S074/S074R03.edf', '/content/drive/MyDrive/files/S074/S074R11.edf', '/content/drive/MyDrive/files/S076/S076R03.edf', '/content/drive/MyDrive/files/S076/S076R11.edf', '/content/drive/MyDrive/files/S076/S076R07.edf', '/content/drive/MyDrive/files/S077/S077R03.edf', '/content/drive/MyDrive/files/S077/S077R07.edf', '/content/drive/MyDrive/files/S077/S077R11.edf', '/content/drive/MyDrive/files/S078/S078R11.edf', '/content/drive/MyDrive/files/S078/S078R07.edf', '/content/drive/MyDrive/files/S078/S078R03.edf', '/content/drive/MyDrive/files/S079/S079R07.edf', '/content/drive/MyDrive/files/S079/S079R03.edf', '/content/drive/MyD

In [ ]:
!pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 21.8 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
if tf.test.gpu_device_name():
    print('Default GPU Device: {}'.format(tf.test.gpu_device_name()))
    !nvidia-smi
else:
    print("Please install GPU version of TF")

Default GPU Device: /device:GPU:0
Wed Mar  4 04:15:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             27W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------

In [ ]:
import mne

raw_list = []

for path in data:
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    raw_list.append(raw)

print("Total raw files loaded:", len(raw_list))

/tmp/ipykernel_252/796524558.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipykernel_252/796524558.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipykernel_252/796524558.py:6: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)


KeyboardInterrupt: 

In [ ]:
t0_all = []
t1_all = []
t2_all = []

for raw in raw_list:

    # 1️⃣ Resample
    raw = raw.copy().resample(160, npad="auto")

    # 2️⃣ Bandpass
    raw = raw.filter(
        l_freq=8.,
        h_freq=30.,
        fir_design='firwin',
        verbose=False
    )

    # 3️⃣ Events
    events, event_dict = mne.events_from_annotations(raw, verbose=False)

    # 4️⃣ Epochs
    epochs = mne.Epochs(
        raw,
        events,
        event_id={'T0': 1, 'T1': 2, 'T2': 3},
        tmin=0.5,
        tmax=2.5,
        baseline=None,
        preload=True,
        verbose=False
    )

    # 5️⃣ Extract numpy data
    data = epochs.get_data()   # shape: (trials, channels, samples)

    # 6️⃣ Normalize per trial (VERY IMPORTANT)
    data = (data - data.mean(axis=2, keepdims=True)) / \
           (data.std(axis=2, keepdims=True) + 1e-8)

    # 7️⃣ Split back per class
    try:
        t0_all.append(data[epochs.events[:,2] == 1])
    except:
        pass

    try:
        t1_all.append(data[epochs.events[:,2] == 2])
    except:
        pass

    try:
        t2_all.append(data[epochs.events[:,2] == 3])
    except:
        pass


# 8️⃣ Concatenate all subjects
t0 = np.concatenate(t0_all, axis=0)
t1 = np.concatenate(t1_all, axis=0)
t2 = np.concatenate(t2_all, axis=0)

print("T0:", t0.shape)
print("T1:", t1.shape)
print("T2:", t2.shape)

In [ ]:
import os

save_path = "/content/drive/MyDrive/BCI_Processed"
os.makedirs(save_path, exist_ok=True)

In [ ]:
import numpy as np

t0 = np.load(f"{save_path}/t0.npy")
t1 = np.load(f"{save_path}/t1.npy")
t2 = np.load(f"{save_path}/t2.npy")

print("T0 shape:", t0.shape)
print("T1 shape:", t1.shape)
print("T2 shape:", t2.shape)

T0 shape: (1687, 64, 321)
T1 shape: (840, 64, 321)
T2 shape: (847, 64, 321)


In [ ]:
import numpy as np

np.save(f"{save_path}/t0.npy", t0)
np.save(f"{save_path}/t1.npy", t1)
np.save(f"{save_path}/t2.npy", t2)

print("Files saved successfully.")

Files saved successfully.


In [ ]:
!pip install pyriemann

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 6.6 MB/s eta 0:00:00


In [ ]:
# Your channel list
ch_names_raw = ['Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.', 'Fc4.', 'Fc6.',
                'C5..', 'C3..', 'C1..', 'Cz..', 'C2..', 'C4..', 'C6..',
                'Cp5.', 'Cp3.', 'Cp1.', 'Cpz.', 'Cp2.', 'Cp4.', 'Cp6.',
                'Fp1.', 'Fpz.', 'Fp2.', 'Af7.', 'Af3.', 'Afz.', 'Af4.', 'Af8.',
                'F7..', 'F5..', 'F3..', 'F1..', 'Fz..', 'F2..', 'F4..', 'F6..', 'F8..',
                'Ft7.', 'Ft8.', 'T7..', 'T8..', 'T9..', 'T10.', 'Tp7.', 'Tp8.',
                'P7..', 'P5..', 'P3..', 'P1..', 'Pz..', 'P2..', 'P4..', 'P6..', 'P8..',
                'Po7.', 'Po3.', 'Poz.', 'Po4.', 'Po8.',
                'O1..', 'Oz..', 'O2..', 'Iz..']

# Clean: remove dots and make uppercase
ch_names = [ch.replace('.', '').upper() for ch in ch_names_raw]

print(ch_names)

['FC5', 'FC3', 'FC1', 'FCZ', 'FC2', 'FC4', 'FC6', 'C5', 'C3', 'C1', 'CZ', 'C2', 'C4', 'C6', 'CP5', 'CP3', 'CP1', 'CPZ', 'CP2', 'CP4', 'CP6', 'FP1', 'FPZ', 'FP2', 'AF7', 'AF3', 'AFZ', 'AF4', 'AF8', 'F7', 'F5', 'F3', 'F1', 'FZ', 'F2', 'F4', 'F6', 'F8', 'FT7', 'FT8', 'T7', 'T8', 'T9', 'T10', 'TP7', 'TP8', 'P7', 'P5', 'P3', 'P1', 'PZ', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO3', 'POZ', 'PO4', 'PO8', 'O1', 'OZ', 'O2', 'IZ']


In [ ]:
motor_channels = ['FC5', 'FC3', 'FC1', 'FCZ', 'FC2', 'FC4', 'FC6', 'C5', 'C3', 'C1', 'CZ', 'C2', 'C4', 'C6', 'CP5', 'CP3', 'CP1', 'CPZ', 'CP2', 'CP4', 'CP6', 'FP1', 'FPZ', 'FP2', 'AF7', 'AF3', 'AFZ', 'AF4', 'AF8', 'F7', 'F5', 'F3', 'F1', 'FZ', 'F2', 'F4', 'F6', 'F8', 'FT7', 'FT8', 'T7', 'T8', 'T9', 'T10', 'TP7', 'TP8', 'P7', 'P5', 'P3', 'P1', 'PZ', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO3', 'POZ', 'PO4', 'PO8', 'O1', 'OZ', 'O2', 'IZ']



In [ ]:
motor_idx = [ch_names.index(ch) for ch in motor_channels]

print("Motor Channel Indices:", motor_idx)
t0.shape

Motor Channel Indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]


(1687, 64, 321)

In [ ]:
t0_motor = t0[:, motor_idx, :]
t1_motor = t1[:, motor_idx, :]
t2_motor = t2[:, motor_idx, :]

print("New shape:", t0_motor.shape)

New shape: (1687, 64, 321)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import copy

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from torch.utils.data import TensorDataset, DataLoader

In [ ]:
y0 = np.zeros(len(t0))
y1 = np.ones(len(t1))
y2 = np.ones(len(t2)) * 2

In [ ]:
X = np.concatenate([t0, t1, t2], axis=0)
y = np.concatenate([y0, y1, y2], axis=0)
X.shape
X = X[:, None, :, :]
X.shape

(3374, 1, 64, 321)

In [ ]:
# spliting data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (2699, 1, 64, 321)
Test shape: (675, 1, 64, 321)


In [ ]:
#normalising
mean = X_train.mean(axis=(0,2), keepdims=True)
std = X_train.std(axis=(0,2), keepdims=True)

X_train = (X_train - mean) / (std + 1e-6)
X_test  = (X_test - mean) / (std + 1e-6)

In [ ]:
#Data augmentation
noise = np.random.normal(0, 0.01, X_train.shape)

X_train_aug = X_train + noise

X_train = np.concatenate([X_train, X_train_aug])
y_train = np.concatenate([y_train, y_train])

print("Augmented Train shape:", X_train.shape)

Augmented Train shape: (5398, 1, 64, 321)


In [ ]:
train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import MDM
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
X_norm = (X - X.mean(axis=2, keepdims=True)) / \
         (X.std(axis=2, keepdims=True) + 1e-8)

models = {
    "Riemann + Logistic": make_pipeline(
        Covariances(estimator='oas'),
        TangentSpace(),
        LogisticRegression(max_iter=2000)
    ),

    "Riemann + SVM": make_pipeline(
        Covariances(estimator='oas'),
        TangentSpace(),
        SVC(kernel='linear')
    ),

    "Riemann + LDA": make_pipeline(
        Covariances(estimator='oas'),
        TangentSpace(),
        LinearDiscriminantAnalysis()
    ),

    "Pure Riemann MDM": make_pipeline(
        Covariances(estimator='oas'),
        MDM()
    ),

    "Riemann + RF": make_pipeline(
        Covariances(estimator='oas'),
        TangentSpace(),
        RandomForestClassifier(n_estimators=200)
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name}: {acc:.3f}")

Riemann + Logistic: 0.609


KeyboardInterrupt: 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

Using device: cuda


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class EEGNet(nn.Module):
    def __init__(self, n_channels=64, n_samples=321, n_classes=2):
        super(EEGNet, self).__init__()

        self.temporal = nn.Conv2d(1, 16, (1, 64), padding=(0,32), bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        self.depthwise = nn.Conv2d(
            16, 32, (n_channels, 1),
            groups=16, bias=False
        )
        self.bn2 = nn.BatchNorm2d(32)

        self.pool1 = nn.AvgPool2d((1, 4))
        self.dropout = nn.Dropout(0.5)

        self.separable = nn.Conv2d(32, 32, (1, 16), padding=(0,8), bias=False)
        self.bn3 = nn.BatchNorm2d(32)
        self.pool2 = nn.AvgPool2d((1, 8))

        self.fc = nn.Linear(20480, n_classes)
    def forward(self, x):
        x = self.temporal(x)
        x = self.bn1(x)
        x = torch.relu(x)

        x = self.depthwise(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.pool1(x)
        x = self.dropout(x)

        x = self.separable(x)
        x = self.bn3(x)
        x = torch.relu(x)
        x = self.pool2(x)
        x = self.dropout(x)

        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [ ]:
model = EEGNet(
    n_channels=X.shape[1],
    n_samples=X.shape[2],
    n_classes=len(np.unique(y))
).to(device)

In [ ]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

In [ ]:
x, y = next(iter(train_loader))
print(x.shape)


torch.Size([64, 1, 64, 321])


In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    patience=5,
    factor=0.5
)

In [ ]:
epochs = 80
best_acc = 0
best_model = copy.deepcopy(model.state_dict())
early_stop_counter = 0

for epoch in range(epochs):

    # TRAIN
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # VALIDATION
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)

            outputs = model(xb)
            _, preds = torch.max(outputs, 1)

            total += yb.size(0)
            correct += (preds == yb).sum().item()

    val_acc = correct / total

    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f} | Val Acc: {val_acc:.4f}")

    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        best_model = copy.deepcopy(model.state_dict())
        early_stop_counter = 0
    else:
        early_stop_counter += 1

Epoch 1 | Loss: 35.9467 | Val Acc: 0.4430
Epoch 2 | Loss: 34.0557 | Val Acc: 0.4444
Epoch 3 | Loss: 33.8404 | Val Acc: 0.4430
Epoch 4 | Loss: 34.5652 | Val Acc: 0.4474
Epoch 5 | Loss: 34.5087 | Val Acc: 0.4430
Epoch 6 | Loss: 33.9206 | Val Acc: 0.4444
Epoch 7 | Loss: 34.4625 | Val Acc: 0.4430
Epoch 8 | Loss: 33.9976 | Val Acc: 0.4474
Epoch 9 | Loss: 34.1575 | Val Acc: 0.4474
Epoch 10 | Loss: 33.5466 | Val Acc: 0.4489
Epoch 11 | Loss: 34.0940 | Val Acc: 0.4444
Epoch 12 | Loss: 33.7614 | Val Acc: 0.4400
Epoch 13 | Loss: 33.8939 | Val Acc: 0.4444
Epoch 14 | Loss: 33.9976 | Val Acc: 0.4459
Epoch 15 | Loss: 33.7609 | Val Acc: 0.4415
Epoch 16 | Loss: 33.7082 | Val Acc: 0.4489
Epoch 17 | Loss: 33.5882 | Val Acc: 0.4415
Epoch 18 | Loss: 33.3379 | Val Acc: 0.4489
Epoch 19 | Loss: 34.3930 | Val Acc: 0.4385
Epoch 20 | Loss: 34.5779 | Val Acc: 0.4459
Epoch 21 | Loss: 33.9381 | Val Acc: 0.4444
Epoch 22 | Loss: 32.6590 | Val Acc: 0.4459
Epoch 23 | Loss: 33.3049 | Val Acc: 0.4459
Epoch 24 | Loss: 33.

In [ ]:
import tensorflow as tf
if tf.test.gpu_device_name():
    print('Default GPU Device: {}'.format(tf.test.gpu_device_name()))
    !nvidia-smi
else:
    print("Please install GPU version of TF")

Default GPU Device: /device:GPU:0
Tue Mar  3 08:36:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             66W /   70W |     379MiB /  15360MiB |     90%      Default |
|                                         |                        |                  N/A |
+-------------